# AIC2026 TEAM-EVAL — Exact Dense Raw Renderer

Required Kaggle inputs:
1. Raw AIC corpus: `/kaggle/input/datasets/nadkli/dataset-aic` (nested roots supported).
2. AI-authored request dataset containing exactly one `anchor_requests.jsonl`: `/kaggle/input/datasets/irthn1311/aic2026-team-eval-anchor-requests` (nested root supported).

Internet required: **Yes, only for cloning the `TRIAGEEG` repository branch**. No model asset or network inference is required. Output ZIP: `/kaggle/working/aic2026_team_eval_dense_bundle.zip`.

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path
from zipfile import ZipFile
DATA_INPUT = Path(os.environ.get('AIC_DATA_ROOT', '/kaggle/input/datasets/nadkli/dataset-aic'))
REPO_URL = os.environ.get('AIC_REPO_URL', 'https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git')
REPO_REF = os.environ.get('AIC_REPO_REF', 'TRIAGEEG')
REPO_DIR = Path(os.environ.get('AIC_REPO_DIR', '/kaggle/working/AIC2026_TeamPTK_SGU'))
REFRESH_REPO = os.environ.get('AIC_REFRESH_REPO', '0') == '1'
ANCHOR_INPUT = Path(os.environ.get('AIC_ANCHOR_REQUEST_ROOT', '/kaggle/input/datasets/irthn1311/aic2026-team-eval-anchor-requests'))
OUTPUT_ROOT = Path('/kaggle/working/aic2026_team_eval_dense')
ZIP_PATH = Path('/kaggle/working/aic2026_team_eval_dense_bundle.zip')
print({'repo_url': REPO_URL, 'repo_ref': REPO_REF, 'repo_dir': str(REPO_DIR), 'data_input': str(DATA_INPUT), 'anchor_input': str(ANCHOR_INPUT), 'output_zip': str(ZIP_PATH), 'internet_required_for_repo_clone': True, 'model_download_required': False})

In [ ]:
if REFRESH_REPO and REPO_DIR.exists():
    if REPO_DIR.parent != Path('/kaggle/working') or REPO_DIR.name != 'AIC2026_TeamPTK_SGU': raise RuntimeError(f'Refusing repository cleanup outside the expected Kaggle path: {REPO_DIR}')
    shutil.rmtree(REPO_DIR)
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
if not (REPO_DIR / 'src/aic2026_eval/pipeline.py').is_file(): raise RuntimeError(f'Cloned repository/ref does not contain TEAM-EVAL package: {REPO_DIR}; push the current TRIAGEEG changes first')
REPO_ROOT = REPO_DIR.resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))
from aic2026_eval.discovery import resolve_dataset_root, resolve_named_file
DATASET_ROOT = resolve_dataset_root(DATA_INPUT)
ANCHOR_REQUESTS = resolve_named_file(ANCHOR_INPUT, 'anchor_requests.jsonl')
BUILD_COMMIT = subprocess.run(['git','rev-parse','HEAD'], cwd=REPO_ROOT, capture_output=True, text=True, check=True).stdout.strip()
print({'resolved_repo': str(REPO_ROOT), 'resolved_dataset': str(DATASET_ROOT), 'resolved_anchor_requests': str(ANCHOR_REQUESTS), 'commit': BUILD_COMMIT})

In [ ]:
from aic2026_eval.io import read_jsonl
REQUESTS = read_jsonl(ANCHOR_REQUESTS)
assert REQUESTS, 'anchor_requests.jsonl must not be empty'
print('anchor_count:', len(REQUESTS), 'modes:', sorted({str(row.get('mode','')).upper() for row in REQUESTS}))

In [ ]:
from aic2026_eval.pipeline import run_dense
RESULT = run_dense(dataset_root=DATASET_ROOT, repository_root=REPO_ROOT, anchor_requests_path=ANCHOR_REQUESTS, output_root=OUTPUT_ROOT, build_commit=BUILD_COMMIT)
assert RESULT['DENSE_RENDERER_STATUS'] == 'READY' and RESULT['rendered_anchor_count'] == len(REQUESTS)
manifest = read_jsonl(OUTPUT_ROOT / 'dense_manifest.jsonl')
assert all(row['frame_identity_exact'] and row['requested_frame_ids'] == row['actual_frame_ids'] for row in manifest)
print(json.dumps(RESULT, indent=2))

In [ ]:
assert Path(RESULT['zip_path']) == ZIP_PATH and ZIP_PATH.is_file()
with ZipFile(ZIP_PATH) as archive: members = archive.namelist()
assert not any(name.endswith(('.mp4','.npy','.npz','.pt','.pth')) for name in members)
print('DENSE_RENDERER_STATUS=READY')
print('SEMANTIC_JUDGMENT_PERFORMED=NO')
print('DOWNLOAD ZIP:', ZIP_PATH, 'size_bytes=', ZIP_PATH.stat().st_size, 'members=', len(members))